# Klassifikation - Finaler Workflow für CatBoost

## Schritt 1 — Datenvorbereitung

In [2]:
# Imports (gekürzt für Dokumentation)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, precision_recall_curve
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier
from pipeline.data_pipeline import preprocessor, X, y

# Damage-Spalte isolieren und entfernen
damage_series = X['damage'].copy()
X = X.drop(columns=['damage'])

# Train/Test-Split mit Stratifikation
X_train, X_test, y_train, y_test, damage_train, damage_test = train_test_split(
    X, y, damage_series, test_size=0.2, random_state=42, stratify=y
)

c:\Users\sofie\Desktop\FernUni Hagen\25 Projektpraktikum\daten\Gruppe 1\project_data_science\models\pipeline\data_pipeline.py:195: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(transactions_for_pipeline['label']):


## Schritt 2 — Custom Objective Function

In [3]:
# Ziel: - Summe Schaden FN + 5 * TP - 10 * FP
def custom_objective_score(y_true, y_pred, damage_series):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()
    fn_mask = (y_true == 1) & (y_pred == 0)
    sum_damage_fn = damage_series[fn_mask].sum()
    score = -sum_damage_fn + (5 * TP) - (10 * FP)
    return float(score)

## Schritt 3 — Randomized Hyperparameter Search mit SMOTE

In [4]:
def randomized_search_with_smote(model_class, param_grid, preprocessor, 
                                  X, y, damage_series, 
                                  model_fixed_params=None, 
                                  n_iter=15, cv_splits=3, random_state=42):
    
    rng = np.random.default_rng(seed=random_state)
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    best_score = -np.inf
    best_params = None
    param_grid_items = list(param_grid.items())

    for iteration in range(n_iter):
        params = {key: rng.choice(list(values)) for key, values in param_grid_items}
        full_params = {**params}
        if model_fixed_params:
            full_params.update(model_fixed_params)

        fold_scores = []

        for train_idx, valid_idx in skf.split(X, y):
            X_train_fold, X_valid_fold = X.iloc[train_idx], X.iloc[valid_idx]
            y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
            damage_valid = damage_series.iloc[valid_idx]

            # Preprocessing und Oversampling
            X_train_prep = preprocessor.fit_transform(X_train_fold)
            X_valid_prep = preprocessor.transform(X_valid_fold)

            smote = SMOTE(sampling_strategy=0.2, random_state=random_state)
            X_resampled, y_resampled = smote.fit_resample(X_train_prep, y_train_fold)

            model = model_class(**full_params)
            model.fit(X_resampled, y_resampled)
            y_pred = model.predict(X_valid_prep)
            score = custom_objective_score(y_valid_fold.reset_index(drop=True), y_pred, damage_valid.reset_index(drop=True))
            fold_scores.append(score)

        mean_score = np.mean(fold_scores)
        if mean_score > best_score:
            best_score = mean_score
            best_params = full_params

    return best_params, best_score

## Schritt 4 — Parameterräume für CatBoost

In [5]:
param_grid_cat = {
    'iterations': [250, 300, 350],
    'learning_rate': [0.07, 0.08, 0.09],
    'depth': [4, 5, 6]
}

## Schritt 5 — Hyperparameter-Optimierung

In [6]:
best_params_cat, best_score_cat = randomized_search_with_smote(
    model_class=CatBoostClassifier,
    param_grid=param_grid_cat,
    preprocessor=preprocessor,
    X=X_train,
    y=y_train,
    damage_series=damage_train,
    model_fixed_params={'verbose': 0, 'random_state': 42}
)

## Schritt 6 — Finales Training auf dem vollständigen Trainingsset

In [7]:
# Preprocessing und Oversampling auf Gesamtdatensatz
X_train_prep = preprocessor.fit_transform(X_train)
smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_prep, y_train)

# Finales Modelltraining
final_cat = CatBoostClassifier(**best_params_cat)
final_cat.fit(X_train_res, y_train_res)


## Schritt 7 — Evaluation auf Testset

In [8]:
# Preprocessing Testset
X_test_prep = preprocessor.transform(X_test)
y_pred = final_cat.predict(X_test_prep)

# Standardmetriken
print(classification_report(y_test, y_pred, target_names=["NORMAL", "FRAUD"]))
print(confusion_matrix(y_test, y_pred))
roc_auc = roc_auc_score(y_test, final_cat.predict_proba(X_test_prep)[:, 1])
print(f"ROC-AUC: {roc_auc:.4f}")

# Custom Objective Score
custom_score = custom_objective_score(
    y_test.reset_index(drop=True), 
    y_pred, 
    damage_test.reset_index(drop=True)
)
print(f"Custom Objective Score (Threshold 0.5): {custom_score:.2f}")

              precision    recall  f1-score   support

      NORMAL       0.97      1.00      0.99     30139
       FRAUD       0.66      0.20      0.31      1000

    accuracy                           0.97     31139
   macro avg       0.82      0.60      0.65     31139
weighted avg       0.96      0.97      0.96     31139

[[30037   102]
 [  799   201]]
ROC-AUC: 0.8403
Custom Objective Score (Threshold 0.5): -5971.86


## Schritt 8 — Threshold-Optimierung

In [9]:
def find_best_threshold(model, preprocessor, X_test, y_test, damage_test, thresholds=np.arange(0.01, 1.0, 0.01)):
    X_test_prep = preprocessor.transform(X_test)
    y_scores = model.predict_proba(X_test_prep)[:, 1]

    best_threshold = 0.5
    best_score = -np.inf

    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)
        score = custom_objective_score(
            y_test.reset_index(drop=True),
            y_pred,
            damage_test.reset_index(drop=True)
        )
        if score > best_score:
            best_score = score
            best_threshold = thresh

    print(f"Optimaler Threshold: {best_threshold:.2f} mit Custom Score: {best_score:.2f}")
    return best_threshold, best_score

best_thresh, best_custom_score = find_best_threshold(
    model=final_cat,
    preprocessor=preprocessor,
    X_test=X_test,
    y_test=y_test,
    damage_test=damage_test
)

Optimaler Threshold: 0.58 mit Custom Score: -5849.64


In [10]:
print(best_params_cat)

{'iterations': 350, 'learning_rate': 0.08, 'depth': 4, 'verbose': 0, 'random_state': 42}


## Ergebnis

Das finale Modell für den CatBoostClassifier konnte durch Hyperparameteroptimierung mit SMOTE sowie anschließender Threshold-Optimierung auf dem Testset die wirtschaftlich gewichtete Zielfunktion weiter optimieren. Trotz der weiterhin herausfordernden Imbalance konnte die Precision auf 66% gesteigert werden, was für die angestrebte Echtzeitanwendung hohe praktische Relevanz besitzt. Der ROC-AUC von 0.84 unterstreicht die gute Trennschärfe des Modells.

- Bester Parameter: {'iterations': 350, 'learning_rate': 0.08, 'depth': 4, 'verbose': 0, 'random_state': 42}
- Custom Objective Score (Threshold 0.5): -5971.86
- Optimaler Threshold: 0.58 mit Custom Score: -5849.64